In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
import pandas as pd

df = pd.read_hdf(
    "metr-la.h5",
    key="df"
)

traffic = df.values

print(traffic.shape)

(34272, 207)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    - sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(34260, 12, 207)
(34260, 207)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
from torch.utils.data import random_split

full_train_dataset = TensorDataset(X_train, y_train)

train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

In [9]:
import torch
import torch.nn as nn

class TemporalConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(3,1),
            padding=(1,0)
        )

    def forward(self,x):

        return torch.relu(
            self.conv(x)
        )

In [10]:
class ClusterHypergraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        num_clusters,
        channels
    ):

        super().__init__()

        self.num_nodes = num_nodes
        self.num_clusters = num_clusters

        self.cluster_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                num_clusters
            )
        )

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self,x):

        H = torch.softmax(
            self.cluster_embeddings,
            dim=1
        )

        hyper_adj = H @ H.T

        hyper_adj = hyper_adj / (
            hyper_adj.sum(
                dim=1,
                keepdim=True
            )
            + 1e-6
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            hyper_adj,
            x
        )

        x = x.permute(
            0,
            2,
            3,
            1
        )

        x = self.weight(x)

        x = x.permute(
            0,
            3,
            1,
            2
        )

        return torch.relu(x)

In [11]:
class CAHSTGCN(nn.Module):

    def __init__(self):

        super().__init__()

        self.temp1 = TemporalConv(1, 64)
        self.temp2 = TemporalConv(64, 64)

        self.hypergraph = ClusterHypergraphConv(
            num_nodes=207,
            num_clusters=32,
            channels=64
        )

        self.fc = nn.Linear(64, 1)

    def forward(self,x):

        x = x.unsqueeze(1)

        x = self.temp1(x)

        x = self.hypergraph(x)

        x = self.temp2(x)

        x = x.mean(dim=2)

        x = x.permute(
            0,
            2,
            1
        )

        x = self.fc(x)

        return x.squeeze(-1)

In [12]:
model = CAHSTGCN()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 207])
torch.Size([64, 207])


In [13]:

model = CAHSTGCN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)

best_val_loss = float('inf')
patience = 15
counter = 0

import time
train_start = time.time()

epochs = 150

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(pred, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    total_loss /= len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:

            pred = model(X_batch)

            loss = criterion(pred, y_batch)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        counter = 0

        torch.save(
            model.state_dict(),
            "best_cah_stgcn.pth"
        )

    else:

        counter += 1

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Train: {total_loss:.6f} "
        f"Val: {val_loss:.6f} "
        f"LR: {optimizer.param_groups[0]['lr']:.6f}"
    )

    if counter >= patience:

        print(f"Early stopping at epoch {epoch+1}")
        break


Epoch 1/150 Train: 0.054341 Val: 0.040766 LR: 0.001000
Epoch 2/150 Train: 0.037000 Val: 0.035185 LR: 0.001000
Epoch 3/150 Train: 0.032084 Val: 0.031251 LR: 0.001000
Epoch 4/150 Train: 0.028099 Val: 0.026944 LR: 0.001000
Epoch 5/150 Train: 0.024707 Val: 0.024163 LR: 0.001000
Epoch 6/150 Train: 0.022434 Val: 0.022106 LR: 0.001000
Epoch 7/150 Train: 0.020445 Val: 0.021008 LR: 0.001000
Epoch 8/150 Train: 0.019291 Val: 0.019317 LR: 0.001000
Epoch 9/150 Train: 0.018315 Val: 0.018851 LR: 0.001000
Epoch 10/150 Train: 0.017502 Val: 0.017650 LR: 0.001000
Epoch 11/150 Train: 0.016669 Val: 0.017073 LR: 0.001000
Epoch 12/150 Train: 0.016155 Val: 0.016384 LR: 0.001000
Epoch 13/150 Train: 0.015832 Val: 0.016360 LR: 0.001000
Epoch 14/150 Train: 0.015549 Val: 0.015826 LR: 0.001000
Epoch 15/150 Train: 0.015363 Val: 0.015724 LR: 0.001000
Epoch 16/150 Train: 0.015085 Val: 0.017191 LR: 0.001000
Epoch 17/150 Train: 0.014922 Val: 0.015479 LR: 0.001000
Epoch 18/150 Train: 0.014770 Val: 0.015563 LR: 0.001000
E

In [14]:
train_time = time.time() - train_start
print("Time Taken:", train_time)

Time Taken: 21322.678713798523


In [ ]:
model.load_state_dict(torch.load("best2_cah_stgcn.pth"))

<All keys matched successfully>

In [16]:
torch.save(
    model.state_dict(),
    "CAH-STGCN-PEMS-BAY.pth"
)

In [17]:
test_dataset = TensorDataset(
    X_test,
    y_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_targets = []

model.eval()

infer_start = time.time()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        pred = model(X_batch)

        all_predictions.append(
            pred.numpy()
        )

        all_targets.append(
            y_batch.numpy()
        )

predictions = np.concatenate(
    all_predictions,
    axis=0
)

infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = np.concatenate(
    all_targets,
    axis=0
)

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

Infer Time: 18.624858856201172
MAE: 0.06876640766859055
RMSE: 0.14102335641858738


In [18]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)

print("MAPE:", mape)
print("R2:", r2)

MAPE: 1.658683e+06
R2: 0.8122178316116333


In [19]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 23457


In [20]:
print("Prediction range:")
print(predictions.min(), predictions.max())

print("Actual range:")
print(true_values.min(), true_values.max())

print("Prediction mean:", predictions.mean())
print("Actual mean:", true_values.mean())

Prediction range:
-0.28768814 1.1281947
Actual range:
0.0 1.0
Prediction mean: 0.7340443
Actual mean: 0.7262074


In [21]:
print(predictions[:10])
print(true_values[:10])

[[0.8964312  0.8970928  0.9493028  ... 0.8786087  0.8577348  0.89754736]
 [0.88553    0.8982644  0.9548271  ... 0.8813509  0.86913705 0.8969991 ]
 [0.8863609  0.9159014  0.94781137 ... 0.8871572  0.8826965  0.9039041 ]
 ...
 [0.9146739  0.9322196  0.9661679  ... 0.8964672  0.8981911  0.931368  ]
 [0.9142506  0.9394082  0.96841073 ... 0.90467596 0.90386856 0.9353833 ]
 [0.91988325 0.9335588  0.9691762  ... 0.90629804 0.9064237  0.9317585 ]]
[[0.88928574 0.7803571  0.9714286  ... 0.93214285 0.91964287 0.9125    ]
 [0.8968254  0.9349206  0.9206349  ... 0.9142857  0.947619   0.9285714 ]
 [0.86190474 0.8142857  0.9809524  ... 0.915873   0.92698413 0.9126984 ]
 ...
 [0.9        0.9        0.96031743 ... 0.9222222  0.94920635 0.9190476 ]
 [0.91071427 0.8464286  0.9785714  ... 0.93392855 0.9464286  0.93214285]
 [0.9589286  0.9125     0.98392856 ... 0.91785717 0.93392855 0.9375    ]]


In [22]:
type(predictions)
predictions.shape

(6852, 207)

In [ ]:
type(true_values)
true_values.shape

(6852, 207)

In [24]:
total_params = sum(p.numel() for p in model.parameters())

print("Parameters:", total_params)

Parameters: 23457
